# MobiHet teacher-first residual / common-base energy
Frozen teacher-first contract; measured 2-round cost gate then same-schedule resume. All learned models DAGSNet. No raw/perturbed/latent/per-record gradients exported. FP32 statistics uplink, coherent paid proxy snapshot, actual FP16 Gaussian factors and FP32 coherent proxy downlink, cohort>=5 trusted emulator, no crypto/DP. Internal group/gp, r20, ten mean-client classification metrics. Stage U; R1/R2 not proven. W&B online authentication before training; private inline key owner-authorized.

In [1]:

import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','wandb==0.29.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
assert wandb.login(key=_wandb_key,anonymous='never',relogin=True,verify=True)
os.environ['WANDB_API_KEY']=_wandb_key
del _wandb_key
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.7 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
opentelemetry-semantic-conventions 0.59b0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.45.1 which is incompatible.
wandb: WARNING The anonymous parameter to wandb.login() has no effect and will be removed in future versions.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] 

W&B authenticated online


In [2]:
CODE=['__init__.py', 'archs.py', 'channel.py', 'dagsnet.py', 'metrics.py', 'mobility.py', 'model_packet.py', 'new_fl.py', 'new_methods.py', 'p2.py', 'pilot.py', 'prepare.py', 'teacher_base.py', 'teacher_driver.py', 'v10_energy.py', 'v10_run.py', 'v2_math.py', 'v2_methods.py', 'v2_run.py', 'v3_dense.py', 'v3_math.py', 'v3_methods.py', 'v3_run.py', 'v4_mala.py', 'v4_run.py', 'v4_schema.py', 'v4_support.py', 'v5_anchor.py', 'v5_books.py', 'v5_run.py', 'v6_geometry.py', 'v6_run.py', 'v6_schema.py', 'v7_basis.py', 'v7_nystrom.py', 'v7_run.py', 'v8_hybrid.py', 'v8_run.py', 'v9_anchor.py', 'v9_run.py']

import hashlib,json,shutil,time
from pathlib import Path
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import torch
count=torch.cuda.device_count()
assert count==2 and all(torch.cuda.get_device_capability(i)==(7,5) for i in range(count))
assert torch.__version__=='2.10.0+cu128',torch.__version__
print('torch',torch.__version__,'gpus',[torch.cuda.get_device_name(i) for i in range(count)],flush=True)
hits=[p.parent for p in Path('/kaggle/input').rglob('v2_run.py') if (p.parent/'train_0_X.npy').exists()]
assert len(hits)==1,hits
SRC=hits[0]; WORK=Path('/kaggle/working/work'); OUT=Path('/kaggle/working/runs')
SOURCE=Path('/kaggle/working/source'); SOURCE.mkdir(exist_ok=True)
for name in ('mh','knowledge','papers/pilot/fixture'): (WORK/name).mkdir(parents=True,exist_ok=True)
manifest=json.loads((SRC/'source_manifest.json').read_text())
for name,sha in manifest.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('source mismatch',name)
    shutil.copyfile(SRC/name,SOURCE/name)
for f in SRC.iterdir():
    if f.name in CODE: dst=WORK/'mh'/f.name
    elif f.name=='meta.json': dst=WORK/'knowledge'/f.name
    elif f.name=='pilot-v9-residual.md': continue
    else: dst=WORK/'papers/pilot/fixture'/f.name
    shutil.copyfile(f,dst)
fixture=json.loads((SRC/'manifest.json').read_text())
for name,sha in fixture['files'].items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,('fixture mismatch',name)
shutil.copyfile(SRC/'source_manifest.json',SOURCE/'source_manifest.json')
shutil.copyfile(SRC/'manifest.json',SOURCE/'fixture_manifest.json')
print('verified source and fixture',len(manifest),len(fixture['files']),flush=True)
sys.path.insert(0,str(WORK))
import numpy as np
from mh.dagsnet import build_model,CFG
from mh.v2_math import Layout,CountSketch,SeedBasis,unpack
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark=False; torch.backends.cudnn.deterministic=True
torch.backends.cuda.matmul.allow_tf32=False
for gpu in range(2):
    device=torch.device(f'cuda:{gpu}'); torch.cuda.set_device(device); torch.manual_seed(42)
    model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True).to(device).eval()
    x=torch.from_numpy(np.load(SRC/'train_0_X.npy')[:32].astype('float32')).to(device)
    y=torch.from_numpy(np.load(SRC/'train_0_y.npy')[:32].astype('int64')).to(device)
    layout=Layout(model); assert layout.size==395024
    with torch.autocast('cuda',dtype=torch.float16): z=model(x)[0].float()
    loss=torch.nn.functional.cross_entropy(z,y); assert torch.isfinite(loss); loss.backward()
    gradient=torch.cat([p.grad.float().reshape(-1) for p in model.parameters()])
    assert torch.isfinite(gradient).all()
    sketch=CountSketch(layout.size,3,256,1701,device)
    packet,delta=sketch.patch(sketch.encode(gradient),128,0); assert torch.isfinite(delta).all()
    assert unpack(packet,device,0)['version']==1
    basis=SeedBasis(model,8,1901,device); coeff=torch.zeros(8,device=device,requires_grad=True)
    torch.nn.functional.cross_entropy(basis.forward(model,coeff,x)[0].float(),y).backward()
    assert torch.isfinite(coeff.grad).all()
    cm=torch.bincount(y*16+z.argmax(1),minlength=256); assert int(cm.sum())==len(y)
    print('T4 real-model deterministic preflight PASS',gpu,flush=True)
    del model,x,y,z,gradient,sketch,basis,coeff,delta; torch.cuda.empty_cache()
torch.cuda.set_device(0)

from mh.model_packet import Codec
for gpu in range(2):
    model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True).to(f'cuda:{gpu}')
    codec=Codec(model,'v8:T4preflight')
    for half in (False,True):
        packet=codec.encode(model.state_dict(),0,1,half=half)
        decoded,base,version=codec.decode(packet,f'cuda:{gpu}',0)
        model.load_state_dict(decoded,strict=True)
        assert version==1
    print('T4 teacher coherentmodel/BN codec PASS',gpu,flush=True)
    del model; torch.cuda.empty_cache()


torch 2.10.0+cu128 gpus ['Tesla T4', 'Tesla T4']
verified source and fixture 42 66
T4 real-model deterministic preflight PASS 0
T4 real-model deterministic preflight PASS 1
T4 teacher coherentmodel/BN codec PASS 0
T4 teacher coherentmodel/BN codec PASS 1


In [3]:
SEED=43
METHODS=['r_dense_ce', 'r64_fl', 'r64_own', 'r64_shuffle', 'r64_classmean', 'r64_hard', 'r64_frozen', 'r512_fl', 'r512_own', 'r512_shuffle', 'r512_classmean', 'r512_hard', 'r512_frozen']
MODULE='mh.v9_run'

T0=time.time(); OUT.mkdir(exist_ok=True)
baseline=METHODS[0]; pending=list(METHODS[1:]); ready=[]; active={}; summary={}; baseline_cost=None
def identity(arm,reason):
    folder=OUT/f'{arm}_s{SEED}';folder.mkdir(exist_ok=True)
    cfg=dict(arm=arm,seed=SEED,private='dagsnet',rounds=20,batch=512,lr=.001,lr_min=.00001,
             clip=1.,eval_every=5,aux_batch=256,aux_weight=1.,temperature=2.,tau=.05,weight_cap=10.,smoke=False)
    data=dict(config=cfg,source={name:manifest[name] for name in CODE},
              fixture=hashlib.sha256((SRC/'manifest.json').read_bytes()).hexdigest())
    data['fingerprint']=hashlib.sha256(json.dumps(data,sort_keys=True).encode()).hexdigest()
    (folder/'config.json').write_text(json.dumps(data,indent=2))
    (folder/'log.txt').write_text(reason+'\n')
def start(arm,gpu,phase):
    folder=OUT/f'{arm}_s{SEED}';folder.mkdir(exist_ok=True)
    env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
    command=[sys.executable,'-m',MODULE,'--arm',arm,'--seed',str(SEED),'--out',str(OUT),
             '--wandb_online','--max_hours','2.0']
    if phase=='probe':command+=['--stop_after','2']
    else:command+=['--resume']
    log=open(folder/'log.txt','a' if phase=='continue' else 'w')
    process=subprocess.Popen(command,cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT)
    active[gpu]=(arm,phase,process,log,time.time())
    print('start',arm,'gpu',gpu,'phase',phase,flush=True)
start(baseline,0,'probe')
while active or pending or ready:
    for gpu in (0,1):
        if gpu in active or time.time()-T0>=4.0*3600:continue
        if ready:
            arm,target_gpu=ready[0]
            if target_gpu!=gpu:continue
            ready.pop(0);start(arm,gpu,'continue')
        elif baseline_cost is not None and pending:
            start(pending.pop(0),gpu,'probe')
    for gpu,(arm,phase,process,log,started) in list(active.items()):
        if process.poll() is None and time.time()-T0>=4.7*3600:process.terminate()
        if process.poll() is None and time.time()-T0>=4.75*3600:process.kill()
        if process.poll() is None:continue
        log.close();del active[gpu]
        folder=OUT/f'{arm}_s{SEED}';path=folder/'result.json'
        result=json.loads(path.read_text()) if path.exists() else {}
        if phase=='probe':
            rows=result.get('history',[])
            if process.returncode or len(rows)!=2:
                summary[arm]=dict(rc=process.returncode or -996,complete=False,cost_pass=None,
                                  reason='cost preflight failed before two complete rounds')
                if arm==baseline:
                    for skipped in pending:identity(skipped,'COST BASELINE MISSING')
                    for skipped in pending:summary[skipped]=dict(rc=-998,complete=False,cost_pass=None)
                    pending.clear()
                continue
            seconds=[row['algorithm_seconds'] for row in rows]
            measured=sum(seconds)/2
            if arm==baseline:baseline_cost=measured
            ratio=measured/baseline_cost;passed=ratio<=2.
            gate=dict(measured_algorithm_seconds=seconds,mean_seconds=measured,
                      baseline_mean_seconds=baseline_cost,ratio=ratio,pass_cost=passed,
                      physical_gpu=gpu,backend='same pinned 2T4; one arm per GPU')
            (folder/'cost_gate.json').write_text(json.dumps(gate,indent=2))
            print('COST',arm,ratio,passed,flush=True)
            if passed:ready.append((arm,gpu))
            else:
                with (folder/'log.txt').open('a') as f:f.write('COST PREFLIGHT REJECT: ratio exceeds 2x fresh dense CE\n')
                summary[arm]=dict(rc=-997,complete=False,cost_pass=False,cost_stopped=True,cost=gate)
        else:
            gate=json.loads((folder/'cost_gate.json').read_text())
            summary[arm]=dict(rc=process.returncode,complete=result.get('complete',False),cost_pass=gate['pass_cost'],cost=gate)
    if not active and time.time()-T0>=4.0*3600:break
    if not active and baseline_cost is None:break
    time.sleep(2)
for arm in METHODS:
    if arm in summary:continue
    folder=OUT/f'{arm}_s{SEED}'
    if not (folder/'config.json').exists():identity(arm,'NOT STARTED: parent budget')
    summary[arm]=dict(rc=-998,complete=False,cost_pass=None,reason='parent budget before arm/continuation')
(OUT/f'summary_s{SEED}.json').write_text(json.dumps(summary,indent=2))
print(summary,flush=True)
shutil.rmtree(WORK)
assert all((v['complete'] and v['rc']==0) or v.get('cost_stopped') for v in summary.values()),'incomplete training outside declared cost stop'


start r_dense_ce gpu 0 phase probe
COST r_dense_ce 1.0 True
start r_dense_ce gpu 0 phase continue
start r64_fl gpu 1 phase probe
COST r64_fl 0.2901456283931369 True
start r64_fl gpu 1 phase continue
start r64_own gpu 1 phase probe
COST r64_own 0.540993453512625 True
start r64_own gpu 1 phase continue
start r64_shuffle gpu 0 phase probe
COST r64_shuffle 0.2986048040174729 True
start r64_shuffle gpu 0 phase continue
start r64_classmean gpu 1 phase probe
COST r64_classmean 0.32824256959880066 True
start r64_classmean gpu 1 phase continue
start r64_hard gpu 0 phase probe
COST r64_hard 0.29917412816557964 True
start r64_hard gpu 0 phase continue
start r64_frozen gpu 1 phase probe
COST r64_frozen 0.3270557934205912 True
start r64_frozen gpu 1 phase continue
start r512_fl gpu 0 phase probe
COST r512_fl 0.301928822423965 True
start r512_fl gpu 0 phase continue
start r512_own gpu 1 phase probe
COST r512_own 0.5658548393160575 True
start r512_own gpu 1 phase continue
start r512_shuffle gpu 0 pha